GPU Check & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))
x = torch.tensor([1.0]).cuda()
print("GPU is healthy:", x)

In [ ]:
import subprocess, sys

# Optional GPU check (safe)
try:
    r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(r.stdout if r.returncode == 0 else " No GPU")
except FileNotFoundError:
    print("nvidia-smi not available (CPU mode)")

# Clean install (modern, stable)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.39.3",
    "datasets",
    "sentencepiece",
    "sacrebleu",
    "accelerate"
], check=True)

print("\n Clean environment ready")

 Compatibility Patches


In [ ]:
import types, sys, pathlib, subprocess

# Patch 1: stub transformers.onnx (removed in transformers >=4.39)
if "transformers.onnx" not in sys.modules:
    _onnx = types.ModuleType("transformers.onnx")
    class OnnxConfig: pass
    class OnnxSeq2SeqConfigWithPast: pass
    def compute_effective_axis_dimension(dimension, fixed_dimension, num_token_to_add=0):
        return (fixed_dimension if dimension == -1 else dimension) + num_token_to_add
    _onnx.OnnxConfig = OnnxConfig
    _onnx.OnnxSeq2SeqConfigWithPast = OnnxSeq2SeqConfigWithPast
    _onnx_utils = types.ModuleType("transformers.onnx.utils")
    _onnx_utils.compute_effective_axis_dimension = compute_effective_axis_dimension
    sys.modules["transformers.onnx"] = _onnx
    sys.modules["transformers.onnx.utils"] = _onnx_utils
    import transformers; transformers.onnx = _onnx
    print("Patch 1: transformers.onnx stub injected")
else:
    print("Patch 1 not needed")

#  Patch 2: relax dependency version checkS
_dep_check = pathlib.Path(
    "/usr/local/lib/python3.12/dist-packages/transformers/dependency_versions_check.py"
)
if _dep_check.exists() and "SKIP =" not in _dep_check.read_text():
    _dep_check.write_text(
        "from .dependency_versions_table import deps\n"
        "from .utils.versions import require_version, require_version_core\n"
        "SKIP = {\"tokenizers\", \"huggingface-hub\", \"accelerate\", \"safetensors\"}\n"
        "pkgs = [\"python\",\"tqdm\",\"regex\",\"requests\",\"packaging\","
        "\"filelock\",\"numpy\",\"huggingface-hub\",\"safetensors\",\"accelerate\",\"pyyaml\"]\n"
        "for pkg in pkgs:\n"
        "    if pkg in deps and pkg not in SKIP:\n"
        "        require_version_core(deps[pkg])\n"
        "def dep_version_check(pkg, hint=None):\n"
        "    require_version(deps[pkg], hint)\n"
    )
    print("Patch 2: dependency version checks relaxed")
else:
    print("Patch 2 not needed")

# Patch 3: fix trainer.py Repository import (removed in huggingface_hub >=1.0)
_trainer = pathlib.Path(
    "/usr/local/lib/python3.12/dist-packages/transformers/trainer.py"
)
if _trainer.exists():
    _ts = _trainer.read_text(encoding="utf-8")
    _old = "from huggingface_hub import Repository, create_repo, upload_folder"
    _new = (
        "from huggingface_hub import create_repo, upload_folder\n"
        "try:\n    from huggingface_hub import Repository\n"
        "except ImportError:\n    Repository = None"
    )
    if "Repository = None" not in _ts and _old in _ts:
        _tmp = pathlib.Path("/tmp/_trainer_p.py")
        _tmp.write_text(_ts.replace(_old, _new), encoding="utf-8")
        subprocess.run(["cp", str(_tmp), str(_trainer)], check=True)
        print(" Patch 3: trainer.py Repository import fixed")
    else:
        print(" Patch 3 not needed")
else:
    print(" Patch 3 skipped (file not found)")

print("\n All patches done — proceed to Cell 3")


In [ ]:
import types, sys, importlib


import transformers.onnx as _onnx_module

class OnnxConfig: pass
class OnnxConfigWithPast: pass
class OnnxSeq2SeqConfigWithPast: pass

def compute_effective_axis_dimension(dimension, fixed_dimension, num_token_to_add=0):
    return (fixed_dimension if dimension == -1 else dimension) + num_token_to_add


_onnx_module.OnnxConfig                 = OnnxConfig
_onnx_module.OnnxConfigWithPast         = OnnxConfigWithPast
_onnx_module.OnnxSeq2SeqConfigWithPast  = OnnxSeq2SeqConfigWithPast


if not hasattr(_onnx_module, "utils"):
    _onnx_utils = types.ModuleType("transformers.onnx.utils")
    _onnx_utils.compute_effective_axis_dimension = compute_effective_axis_dimension
    sys.modules["transformers.onnx.utils"] = _onnx_utils
else:
    _onnx_module.utils.compute_effective_axis_dimension = compute_effective_axis_dimension


if "transformers.models.bart.configuration_bart" in sys.modules:
    del sys.modules["transformers.models.bart.configuration_bart"]

print("ONNX patch applied")

In [ ]:
import subprocess, sys, os

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.39.3",
    "accelerate==0.28.0",
    "peft==0.10.0",
    "datasets",
    "sacrebleu",
    "sacremoses",
    "sentencepiece",
    "ftfy",
], check=True)


mods = [k for k in sys.modules if "peft" in k]
for m in mods:
    del sys.modules[m]

import peft
print(f"peft version: {peft.__version__}")
print("Packages installed")

In [ ]:
import types, sys

if "transformers.onnx" in sys.modules:
    _onnx = sys.modules["transformers.onnx"]
else:
    _onnx = types.ModuleType("transformers.onnx")
    sys.modules["transformers.onnx"] = _onnx

class OnnxConfig: pass
class OnnxConfigWithPast: pass
class OnnxSeq2SeqConfigWithPast: pass
class PatchingSpec: pass

_onnx.OnnxConfig = OnnxConfig
_onnx.OnnxConfigWithPast = OnnxConfigWithPast
_onnx.OnnxSeq2SeqConfigWithPast = OnnxSeq2SeqConfigWithPast
_onnx.PatchingSpec = PatchingSpec

import transformers
transformers.onnx = _onnx
print("ONNX stub ready")

import sacrebleu, shutil, gc, os
from transformers import (
    MarianMTModel, MarianTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    EarlyStoppingCallback, TrainerCallback,
)


## Cell 3 — HuggingFace Login

In [46]:
from huggingface_hub import login

login()  # will show a prompt to enter token interactively
print("Logged in to HuggingFace")

Logged in to HuggingFace


Configuration


In [ ]:
import os, re, logging
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
logging.basicConfig(level=logging.WARNING)

LANGUAGES         = ["tamil", "kannada"]
MAX_TRAIN_SAMPLES = 20_000
MAX_EVAL_SAMPLES  = 200
MAX_SEQ_LEN       = 128
LEARNING_RATE     = 2e-5
BATCH_SIZE        = 16
GRAD_ACCUM_STEPS  = 4
NUM_EPOCHS        = 5
WARMUP_STEPS      = 100
WEIGHT_DECAY      = 0.01
TRAIN_SPLIT       = 0.95

# MarianMT model IDs for each language
MODEL_ID = {
    "tamil":   "Helsinki-NLP/opus-mt-mul-en",
    "kannada": "Helsinki-NLP/opus-mt-mul-en",
}

BASE_DIR       = Path("/content/vividh_translation")
DATA_DIR       = BASE_DIR / "data"
CACHE_DIR      = BASE_DIR / "cache"
CHECKPOINT_DIR = BASE_DIR / "checkpoints"
OUTPUT_DIR     = BASE_DIR / "output"
for d in (DATA_DIR, CACHE_DIR, CHECKPOINT_DIR, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Config ready | Device: {DEVICE.upper()}")
print(f"  Languages: {LANGUAGES}")

 Craft Keywords & Seed Pairs

In [ ]:
CRAFT_KEYWORDS = {
    "tamil": [
        "புடவை", "பட்டு", "மண்பாண்டம்", "கூடை", "கைவினை",
        "நெசவு", "தறி", "கலசம்", "மண்", "தையல்", "வேலைப்பாடு",
        "சிற்பம்", "செம்பு", "பித்தளை", "சந்தன",
    ],
    "kannada": [
        "ಸೀರೆ", "ರೇಷ್ಮೆ", "ಮಡಕೆ", "ಬುಟ್ಟಿ", "ಕರಕುಶಲ",
        "ಕೈಮಗ್ಗ", "ನೇಯ್ಗೆ", "ಕಸೂತಿ", "ಬಿದಿರು", "ಮಣ್ಣು",
        "ತಾಮ್ರ", "ಬೆಳ್ಳಿ", "ಮರ", "ಗೊಂಬೆ", "ಚಿತ್ರ",
    ],
}

CRAFT_SEED_PAIRS = {
    "tamil": [
        ("இந்த புடவை கையால் நெய்யப்பட்டது",                  "This saree is hand-woven"),
        ("கஞ்சிவரம் பட்டு புடவை தங்க நூலில் செய்யப்பட்டது",  "Kanjivaram silk saree is made with gold thread"),
        ("இந்த மண்பாண்டம் சுடுமண்ணால் செய்யப்பட்டது",        "This pottery is made from fired clay"),
        ("கையால் பின்னப்பட்ட மூங்கில் கூடை",                 "Hand-woven bamboo basket"),
        ("இயற்கை சாயம் பயன்படுத்தி நெய்யப்பட்ட துணி",       "Fabric woven using natural dyes"),
        ("கைத்தறியில் செய்யப்பட்ட பருத்தி புடவை",           "Cotton saree made on a handloom"),
        ("தோட்டி வேலையில் செய்யப்பட்ட வெள்ளி நகை",          "Silver jewellery made with filigree work"),
        ("பாரம்பரிய முறையில் செய்யப்பட்ட மண் தட்டு",        "Clay plate made in the traditional style"),
        ("சந்தன மரத்தால் செய்யப்பட்ட கடவுள் சிலை",          "God idol carved from sandalwood"),
        ("கையால் வரையப்பட்ட தஞ்சாவூர் ஓவியம்",              "Hand-painted Thanjavur painting"),
        ("நடுவில் வண்ண வேலைப்பாடு உள்ள கூடை",               "Basket with colourful patterns in the centre"),
        ("இந்த பானை சுடு மண்ணால் உருவாக்கப்பட்டது",         "This pot is created from kiln-fired clay"),
    ],
    "kannada": [
        ("ಈ ಸೀರೆಯನ್ನು ಕೈಯಿಂದ ನೇಯಲಾಗಿದೆ",                   "This saree is hand-woven"),
        ("ಮೈಸೂರು ರೇಷ್ಮೆ ಸೀರೆ ಚಿನ್ನದ ಜರಿಯಿಂದ ತಯಾರಿಸಲಾಗಿದೆ", "Mysore silk saree is made with gold zari"),
        ("ಈ ಮಡಕೆಯನ್ನು ಮಣ್ಣಿನಿಂದ ತಯಾರಿಸಲಾಗಿದೆ",              "This pot is made from clay"),
        ("ಕೈಯಿಂದ ಹೆಣೆದ ಬಿದಿರಿನ ಬುಟ್ಟಿ",                     "Hand-woven bamboo basket"),
        ("ನೈಸರ್ಗಿಕ ಬಣ್ಣ ಬಳಸಿ ನೇದ ಬಟ್ಟೆ",                    "Fabric woven using natural dyes"),
        ("ಕೈಮಗ್ಗದಲ್ಲಿ ತಯಾರಿಸಿದ ಹತ್ತಿ ಸೀರೆ",                 "Cotton saree made on a handloom"),
        ("ಬೆಳ್ಳಿ ಫಿಲಿಗ್ರಿ ಕೆಲಸದ ಆಭರಣ",                       "Silver filigree jewellery"),
        ("ಸಾಂಪ್ರದಾಯಿಕ ವಿಧಾನದಲ್ಲಿ ತಯಾರಿಸಿದ ಮಣ್ಣಿನ ತಟ್ಟೆ",   "Clay plate made in the traditional style"),
        ("ಶ್ರೀಗಂಧ ಮರದಿಂದ ತಯಾರಿಸಿದ ದೇವರ ವಿಗ್ರಹ",             "God idol carved from sandalwood"),
        ("ಕೈಯಿಂದ ಚಿತ್ರಿಸಿದ ಮೈಸೂರು ಚಿತ್ರಕಲೆ",               "Hand-painted Mysore style painting"),
        ("ಮಧ್ಯದಲ್ಲಿ ಬಣ್ಣ ವಿನ್ಯಾಸ ಹೊಂದಿರುವ ಬುಟ್ಟಿ",           "Basket with colourful patterns in the centre"),
        ("ಈ ಮಡಕೆಯನ್ನು ಬಿಡಿಸಿದ ಮಣ್ಣಿನಿಂದ ತಯಾರಿಸಲಾಗಿದೆ",       "This pot is created from kiln-fired clay"),
    ],
}

print(f"Craft data ready — {len(CRAFT_SEED_PAIRS[LANGUAGES[0]])} seed pairs for {LANGUAGES[0]}")


Data Preparation


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.39.3",
    "accelerate==0.28.0",
    "datasets",
    "sacrebleu",
    "sacremoses",
    "sentencepiece",
    "ftfy",
], check=True)
print("Packages installed")

In [ ]:
from datasets import Dataset, DatasetDict, load_dataset, load_from_disk
import ftfy

def load_and_preprocess(language, max_samples):
    lang_code  = "ta" if language == "tamil" else "kn"
    cache_path = DATA_DIR / f"processed_{language}"

    if cache_path.exists():
        print(f"  Loading {language} from cache...")
        return load_from_disk(str(cache_path))

    print(f"  Downloading {language} ({lang_code}) — {max_samples:,} pairs...")
    ds = load_dataset(
        "ai4bharat/samanantar", lang_code,
        split=f"train[:{max_samples}]",
        cache_dir=str(CACHE_DIR)
    )
    if "src" in ds.column_names: ds = ds.rename_column("src", "source")
    if "tgt" in ds.column_names: ds = ds.rename_column("tgt", "target")

    df = ds.to_pandas()
    print(f"  Before filter: {len(df):,}")

    # Fix encoding and clean
    for col in ["source", "target"]:
        df[col] = df[col].apply(lambda x: ftfy.fix_text(str(x)).strip())
        df[col] = df[col].str.replace(r'\s+', ' ', regex=True)
        df[col] = df[col].str.replace(r'http\S+|www\S+', '', regex=True)
        df[col] = df[col].str.replace(r'<[^>]+>', '', regex=True)
        df[col] = df[col].str.strip()

    # Filter bad samples
    mask = (
        df["source"].str.split().str.len().between(3, 80) &
        df["target"].str.split().str.len().between(3, 80) &
        (df["source"] != df["target"])
    )
    df = df[mask].reset_index(drop=True)
    print(f"  After filter : {len(df):,}")

    # Swap: Samanantar has source=English, target=Indic
    # We need source=Indic, target=English
    df = df.rename(columns={"source": "target", "target": "source"})

    print(f"  SRC ({language}): {df['source'].iloc[0][:60]}")
    print(f"  TGT (english) : {df['target'].iloc[0][:60]}")

    # Train/val split
    split_idx = int(len(df) * TRAIN_SPLIT)
    result = DatasetDict({
        "train":      Dataset.from_pandas(df[:split_idx].reset_index(drop=True)),
        "validation": Dataset.from_pandas(df[split_idx:].reset_index(drop=True)),
    })

    # Add craft seed pairs to boost domain vocabulary
    craft_pairs = CRAFT_SEED_PAIRS.get(language, [])
    if craft_pairs:
        craft_df = pd.DataFrame(craft_pairs, columns=["source", "target"])
        craft_df = pd.concat([craft_df] * 50, ignore_index=True)
        train_df = pd.DataFrame({
            "source": result["train"]["source"],
            "target": result["train"]["target"]
        })
        combined = pd.concat([train_df, craft_df], ignore_index=True).sample(
            frac=1, random_state=42).reset_index(drop=True)
        result["train"] = Dataset.from_pandas(combined)
        print(f"  Added craft pairs → Train: {len(result['train']):,}")

    result.save_to_disk(str(cache_path))
    return result

print("Data functions ready")

In [ ]:
import shutil
from pathlib import Path

DATA_DIR = Path("/content/vividh_translation/data")
for lang in ["tamil", "kannada"]:
    cache = DATA_DIR / f"processed_{lang}"
    if cache.exists():
        shutil.rmtree(cache)
        print(f"Cleared {lang} cache")

Tokenization and fine tuning

In [ ]:
import types, sys, sacrebleu, shutil, gc, os
import numpy as np
import torch


if "transformers.onnx" in sys.modules:
    _onnx = sys.modules["transformers.onnx"]
else:
    _onnx = types.ModuleType("transformers.onnx")
    sys.modules["transformers.onnx"] = _onnx

class OnnxConfig: pass
class OnnxConfigWithPast: pass
class OnnxSeq2SeqConfigWithPast: pass
class PatchingSpec: pass

_onnx.OnnxConfig = OnnxConfig
_onnx.OnnxConfigWithPast = OnnxConfigWithPast
_onnx.OnnxSeq2SeqConfigWithPast = OnnxSeq2SeqConfigWithPast
_onnx.PatchingSpec = PatchingSpec

import transformers
transformers.onnx = _onnx

from transformers import (
    MarianMTModel, MarianTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    EarlyStoppingCallback, TrainerCallback,
)
from google.colab import drive
drive.mount('/content/drive')

current_language = None

class DriveSaveCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        try:
            save_path = f'/content/drive/MyDrive/vividh_checkpoints/{current_language}/epoch_{int(state.epoch)}'
            os.makedirs(save_path, exist_ok=True)
            kwargs["model"].save_pretrained(save_path)
            kwargs["tokenizer"].save_pretrained(save_path)
            print(f"Epoch {int(state.epoch)} saved to Drive")
        except Exception as e:
            print(f"Drive save failed: {e}")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple): preds = preds[0]
    preds  = np.where(preds  != -100, preds,  tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    pred_str  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    label_str = tokenizer.batch_decode(labels, skip_special_tokens=True)
    bleu = sacrebleu.corpus_bleu(pred_str, [label_str])
    chrf = sacrebleu.corpus_chrf(pred_str, [label_str])
    return {"bleu": round(bleu.score, 2), "chrf": round(chrf.score, 2)}

SANITY_TESTS = {
    "tamil":   [
        "என் பெயர் ஹர்ஷா",
        "இந்த புடவை கையால் நெய்யப்பட்டது",
        "வணக்கம்",
    ],
    "kannada": [
        "ನನ್ನ ಹೆಸರು ಹರ್ಷಾ",
        "ಈ ಸೀರೆಯನ್ನು ಕೈಯಿಂದ ನೇಯಲಾಗಿದೆ",
        "ನಮಸ್ಕಾರ",
    ],
}

for current_language in LANGUAGES:
    print(f"\n{'='*60}")
    print(f"  Training {current_language.upper()} → English")
    print(f"{'='*60}")

    # Load and preprocess data
    datasets = load_and_preprocess(current_language, MAX_TRAIN_SAMPLES)
    print(f"  Train: {len(datasets['train']):,}  Val: {len(datasets['validation']):,}")

    # Load tokenizer and model — use MODEL_IDS dict
    model_id  = MODEL_IDS[current_language]
    tokenizer = MarianTokenizer.from_pretrained(model_id)
    model     = MarianMTModel.from_pretrained(model_id).to(DEVICE)

    # Tokenize dataset
    def tokenize_fn(batch):
        inputs = tokenizer(
            batch["source"],
            max_length=MAX_SEQ_LEN, truncation=True, padding=False
        )
        with tokenizer.as_target_tokenizer():
            targets = tokenizer(
                batch["target"],
                max_length=MAX_SEQ_LEN, truncation=True, padding=False
            )
        inputs["labels"] = [
            [-100 if t == tokenizer.pad_token_id else t for t in tgt]
            for tgt in targets["input_ids"]
        ]
        return inputs

    tokenized = datasets.map(
        tokenize_fn, batched=True, batch_size=1000,
        remove_columns=["source", "target"],
        desc=f"Tokenising {current_language}",
    )
    if len(tokenized["validation"]) > MAX_EVAL_SAMPLES:
        tokenized["validation"] = tokenized["validation"].select(range(MAX_EVAL_SAMPLES))

    print(f"Tokenized — Train: {len(tokenized['train']):,}  Val: {len(tokenized['validation']):,}")

    # Training arguments
    training_args = Seq2SeqTrainingArguments(
        output_dir                  = str(CHECKPOINT_DIR / current_language),
        num_train_epochs            = NUM_EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM_STEPS,
        learning_rate               = LEARNING_RATE,
        weight_decay                = WEIGHT_DECAY,
        warmup_steps                = WARMUP_STEPS,
        fp16                        = True,
        max_grad_norm               = 1.0,
        evaluation_strategy         = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "bleu",
        greater_is_better           = True,
        predict_with_generate       = True,
        generation_max_length       = MAX_SEQ_LEN,
        generation_num_beams        = 4,
        logging_steps               = 100,
        report_to                   = "none",
        dataloader_num_workers      = 2,
        save_total_limit            = 3,
        seed                        = 42,
    )

    trainer = Seq2SeqTrainer(
        model           = model,
        args            = training_args,
        train_dataset   = tokenized["train"],
        eval_dataset    = tokenized["validation"],
        tokenizer       = tokenizer,
        data_collator   = DataCollatorForSeq2Seq(
            tokenizer, model=model,
            label_pad_token_id=-100, pad_to_multiple_of=8
        ),
        compute_metrics = compute_metrics,
        callbacks       = [
            EarlyStoppingCallback(early_stopping_patience=2),
            DriveSaveCallback(),
        ],
    )

    print(f"\n  Fine-tuning {current_language.upper()} | MarianMT")
    print(f"  Effective batch : {BATCH_SIZE * GRAD_ACCUM_STEPS}")
    print(f"  Train samples   : {len(tokenized['train']):,}")
    print("  " + "="*48)

    try:
        result = trainer.train()
        print(f"\n{current_language.upper()} training complete!")
        print(f"     Loss : {result.training_loss:.4f}")
        print(f"     Steps: {result.global_step}")
    except Exception as e:
        if "early stopping" in str(e).lower():
            print(f"\n Early stopping triggered — best model saved")
        else:
            raise e

    # Final save to Drive
    final_path = f'/content/drive/MyDrive/vividh_checkpoints/{current_language}/final'
    os.makedirs(final_path, exist_ok=True)
    model.save_pretrained(final_path)
    tokenizer.save_pretrained(final_path)
    print(f"Final model saved to Drive")

    # Sanity check
    print(f"\n  Sanity check [{current_language.upper()} → English]:")
    model.eval()
    for t in SANITY_TESTS[current_language]:
        inputs = tokenizer(t, return_tensors="pt", truncation=True).to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, num_beams=4, max_length=64)
        translation = tokenizer.decode(out[0], skip_special_tokens=True)
        print(f"    IN : {t}")
        print(f"    OUT: {translation}")
        print()

    # Clear GPU for next language
    del model, trainer, tokenized, datasets
    gc.collect()
    torch.cuda.empty_cache()
    print(f"GPU cleared for next language\n")

print("Both Tamil and Kannada trained and saved to Drive!")

Translate Your Own Sentences

In [ ]:
import subprocess, sys
subprocess.run(["apt-get", "install", "-y", "portaudio19-dev"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "SpeechRecognition", "pyaudio", "-q"], check=True)

import ipywidgets as widgets
from IPython.display import display, clear_output
import speech_recognition as sr
from transformers import MarianMTModel, MarianTokenizer
import torch, os

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Find best checkpoint for each language
def find_best_checkpoint(language):
    base = f"/content/drive/MyDrive/vividh_checkpoints/{language}"
    # Try final first, then epoch folders
    if os.path.exists(f"{base}/final"):
        return f"{base}/final"
    # Find latest epoch folder
    epochs = sorted([
        d for d in os.listdir(base) if d.startswith("epoch_")
    ], reverse=True)
    if epochs:
        return f"{base}/{epochs[0]}"
    return base

# Load both models
print("Loading Tamil model...")
tamil_path = find_best_checkpoint("tamil")
print(f"  Using: {tamil_path}")
tamil_tokenizer = MarianTokenizer.from_pretrained(tamil_path)
tamil_model = MarianMTModel.from_pretrained(tamil_path).to(DEVICE)
tamil_model.eval()

print("Loading Kannada model...")
kannada_path = find_best_checkpoint("kannada")
print(f"  Using: {kannada_path}")
kannada_tokenizer = MarianTokenizer.from_pretrained(kannada_path)
kannada_model = MarianMTModel.from_pretrained(kannada_path).to(DEVICE)
kannada_model.eval()

print("Both models loaded")

def detect_language(text):
    tamil_chars   = sum(1 for c in text if '\u0B80' <= c <= '\u0BFF')
    kannada_chars = sum(1 for c in text if '\u0C80' <= c <= '\u0CFF')
    if tamil_chars > kannada_chars:
        return "tamil"
    elif kannada_chars > tamil_chars:
        return "kannada"
    return "unknown"

def translate(text):
    lang = detect_language(text)
    if lang == "tamil":
        tok, mdl = tamil_tokenizer, tamil_model
    elif lang == "kannada":
        tok, mdl = kannada_tokenizer, kannada_model
    else:
        return "Could not detect language — please type Tamil or Kannada text", "unknown"
    inputs = tok(text, return_tensors="pt", truncation=True, max_length=128).to(DEVICE)
    with torch.no_grad():
        out = mdl.generate(**inputs, num_beams=4, max_length=128)
    return tok.decode(out[0], skip_special_tokens=True), lang

# UI
text_input    = widgets.Textarea(
    placeholder="Type Tamil or Kannada sentence here, or use voice...",
    layout=widgets.Layout(width="600px", height="80px")
)
lang_label    = widgets.Label(value="Language: Auto-detect")
voice_btn     = widgets.Button(description="🎤 Record Voice",  button_style="info")
translate_btn = widgets.Button(description="🔄 Translate",     button_style="success")
clear_btn     = widgets.Button(description="🗑 Clear",         button_style="warning")
output_area   = widgets.Output()

def on_voice_click(b):
    with output_area:
        clear_output()
        print("🎤 Listening... Speak now!")
        r = sr.Recognizer()
        try:
            with sr.Microphone() as source:
                r.adjust_for_ambient_noise(source, duration=1)
                audio = r.listen(source, timeout=5)
            # Try Tamil first, then Kannada
            recognized = None
            for lang_code in ["ta-IN", "kn-IN"]:
                try:
                    recognized = r.recognize_google(audio, language=lang_code)
                    break
                except:
                    continue
            if recognized:
                text_input.value = recognized
                print(f"✅ Recognized: {recognized}")
            else:
                print("❌ Could not recognize speech. Try again.")
        except sr.WaitTimeoutError:
            print("⏱ No speech detected. Try again.")
        except sr.UnknownValueError:
            print("❌ Could not understand audio. Try again.")
        except Exception as e:
            print(f"❌ Microphone error: {e}")
            print("💡 Tip: Use text input if microphone is unavailable in Colab.")

def on_translate_click(b):
    with output_area:
        clear_output()
        sentence = text_input.value.strip()
        if not sentence:
            print("⚠️ Please type or record a sentence first.")
            return
        print("🔄 Translating...")
        result, lang = translate(sentence)
        lang_label.value = f"Language detected: {lang.capitalize()}"
        print(f"\n   IN  ({lang.capitalize()}) : {sentence}")
        print(f"   OUT (English)        : {result}")

def on_clear_click(b):
    text_input.value = ""
    lang_label.value = "Language: Auto-detect"
    with output_area:
        clear_output()

voice_btn.on_click(on_voice_click)
translate_btn.on_click(on_translate_click)
clear_btn.on_click(on_clear_click)

print("\n Tamil / Kannada → English Translator")
print("=" * 45)
display(lang_label)
display(text_input)
display(widgets.HBox([voice_btn, translate_btn, clear_btn]))
display(output_area)